In [1]:
!pip install datasets
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "

In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.0 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch
from jiwer import wer
import evaluate


In [4]:
librispeech = load_dataset("RaphaelOlivier/librispeech_asr_adversarial", "adv", split='natural')

README.md:   0%|          | 0.00/2.65k [00:00<?, ?B/s]

librispeech_asr_adversarial.py:   0%|          | 0.00/5.59k [00:00<?, ?B/s]

The repository for RaphaelOlivier/librispeech_asr_adversarial contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/RaphaelOlivier/librispeech_asr_adversarial.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating natural split: 0 examples [00:00, ? examples/s]

Generating adv_0.04 split: 0 examples [00:00, ? examples/s]

Generating adv_0.015 split: 0 examples [00:00, ? examples/s]

Generating adv_0.015_RIR split: 0 examples [00:00, ? examples/s]

In [5]:
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h")
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")

config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

In [6]:
import IPython.display as ipd
example = librispeech[10]

audio_array = example['audio']['array']

display(ipd.Audio(audio_array, rate=16000))
print(example["true_text"])



IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


In [7]:
from utils import transcribe_audio

In [8]:
predicted_transcription = transcribe_audio(audio_array, 16000, processor, model)
print(predicted_transcription)

IT IS THIS THAT IS OF INTEREST TO THEORY OF KNOWLEDGE


In [ ]:
from jiwer import wer
import evaluate
cer_metric = evaluate.load("cer")
import json


# Extract audio arrays, sampling rates, and ground truths
audio_arrays = [example["audio"]["array"] for example in librispeech]
sampling_rates = [example["audio"]["sampling_rate"] for example in librispeech]
ground_truths = [example["true_text"].lower().strip() for example in librispeech]

# Generate transcriptions for all samples
try:
    transcriptions = [
        transcribe_audio(audio_array, sampling_rate, processor, model).lower().strip()
        for audio_array, sampling_rate in zip(audio_arrays, sampling_rates)
    ]
except Exception as e:
    print(f"Error during batch transcription: {e}")
    transcriptions = []


In [ ]:
import evaluate

# load both metrics
wer_metric = evaluate.load("wer")    # Word‑Error‑Rate :contentReference[oaicite:0]{index=0}
cer_metric = evaluate.load("cer")


# compute them in one shot
avg_wer = wer_metric.compute(predictions=transcriptions, references=ground_truths)
avg_cer = cer_metric.compute(predictions=transcriptions, references=ground_truths)

print(f"Average WER: {avg_wer:.4f} ({avg_wer*100:.2f}%)")
print(f"Average CER: {avg_cer:.4f} ({avg_cer*100:.2f}%)")


Average WER: 0.0290 (2.90%)
Average CER: 0.0082 (0.82%)


In [ ]:
from cramer_ipm import cramer_ipm_attack

In [13]:
from utils import transcribe_audio, preprocess_audio, calculate_snr , LibriSpeechDataset, custom_collate_fn

In [ ]:
# Load metrics
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

# Select example
example = librispeech[11]
audio_array = example["audio"]["array"]  # Raw audio waveform
ground_truth = example["true_text"]      # Ground truth transcription
target_transcription = "HELLO WORLD"     # Target transcription
# Run the Cramér-IPM attack
adversarial_waveform = cramer_ipm_attack(
    audio_array=audio_array,
    ground_truth=ground_truth,
    target_transcription=target_transcription,
    model=model,
    processor=processor,
    epsilon=0.001,
    num_iterations=10,
    lambda_ipm=1
)

original_transcription = transcribe_audio(audio_array, 16000, processor, model)
adversarial_transcription = transcribe_audio(adversarial_waveform, 16000, processor, model)


# Calculate CER and WER
cer_original = cer_metric.compute(predictions=[original_transcription], references=[ground_truth])
wer_original = wer_metric.compute(predictions=[original_transcription], references=[ground_truth])
cer = cer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])
wer = wer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])

# Display audio
print("Original Audio:")
display(ipd.Audio(audio_array, rate=16000))
print("Adversarial Audio:")
display(ipd.Audio(adversarial_waveform, rate=16000))

# Print transcription and metrics
print(f"Ground Truth: {ground_truth}")
print(f"Original Transcription: {original_transcription}")
print(f"Adversarial Transcription: {adversarial_transcription}")
print(f"Original CER: {cer_original:.2f}")
print(f"Original WER: {wer_original:.2f}")
print(f"Adversarial CER: {cer:.2f}")
print(f"Adversarial WER: {wer:.2f}")

Original Audio:


Adversarial Audio:


Ground Truth: AS USED IN THE SPEECH OF EVERYDAY LIFE THE WORD CARRIES AN UNDERTONE OF DEPRECATION
Original Transcription: AS USED IN THE SPEECH OF EVERYDAY LIFE THE WORD CARRIES AN UNDERTONE OF DEPRECATION
Adversarial Transcription: AS USIN A SPEECH OF EVERY DAY LIFE THE GOD CARIS AN UNDERTINATON
Original CER: 0.00
Original WER: 0.00
Adversarial CER: 0.29
Adversarial WER: 0.67


In [25]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import IPython.display as ipd
import evaluate
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from utils import transcribe_audio, preprocess_audio, calculate_snr, LibriSpeechDataset, custom_collate_fn


# Main loop with parallelization
dataset = LibriSpeechDataset(librispeech, processor)
dataloader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,  # Adjust based on system
    collate_fn=custom_collate_fn
)

device = "cuda" if torch.cuda.is_available() else "cpu"
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

# Ensure model is on the correct device
model.to(device)

# Define epsilon values to test
epsilon_values = [0.001, 0.01, 0.1]
target_transcription = "HELLO WORLD"
selected_indices = [0, 1, 2]

# Evaluate for each epsilon
for epsilon in epsilon_values:
    print(f"\n=== Epsilon: {epsilon} ===")

    cer_list = []
    wer_list = []
    snr_list = []
    demo_samples = []

    for batch_audio, batch_ground_truth, batch_indices in dataloader:
        batch_indices = batch_indices.tolist()

        # Run Cramér-IPM attack on batch
        adversarial_waveforms,results = cramer_ipm_attack(
            audio_tensors=batch_audio,
            target_transcription=target_transcription,
            model=model,
            processor=processor,
            epsilon=epsilon,
            num_iterations=10,
            lambda_ipm=0.1,
            device=device
        )

        # Process each sample in batch
        for i, (audio_tensor, adv_waveform, ground_truth, idx) in enumerate(zip(batch_audio, adversarial_waveforms, batch_ground_truth, batch_indices)):
            audio_array = audio_tensor.numpy()
            adv_waveform = adv_waveform.numpy()

            # Transcribe audio
            original_transcription = transcribe_audio(audio_array, 16000, processor, model)
            adversarial_transcription = transcribe_audio(adv_waveform, 16000, processor, model)

            # Compute metrics
            cer = cer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])
            wer = wer_metric.compute(predictions=[adversarial_transcription], references=[ground_truth])
            snr = calculate_snr(audio_array, adv_waveform)

            cer_list.append(cer)
            wer_list.append(wer)
            snr_list.append(snr)

            # Store info for demo samples
            if idx in selected_indices:
                demo_samples.append({
                    'original_audio': audio_array,
                    'adversarial_audio': adv_waveform,
                    'original_transcription': original_transcription,
                    'adversarial_transcription': adversarial_transcription,
                    'ground_truth': ground_truth,
                    'cer': cer,
                    'wer': wer,
                    'snr': snr
                })

        print(f"Processed batch, total samples: {len(cer_list)}")

    # Compute and print overall metrics
    overall_cer = np.mean(cer_list)
    overall_wer = np.mean(wer_list)
    overall_snr = np.mean(snr_list)
    print(f"Overall CER: {overall_cer:.2f}")
    print(f"Overall WER: {overall_wer:.2f}")
    print(f"Overall SNR: {overall_snr:.2f} dB")

    # Display demo samples
    for i, sample in enumerate(demo_samples):
        print(f"\nSample {i}:")
        print(f"Ground Truth: {sample['ground_truth']}")
        print(f"Original Transcription: {sample['original_transcription']}")
        print(f"Adversarial Transcription: {sample['adversarial_transcription']}")
        print(f"CER: {sample['cer']:.2f}")
        print(f"WER: {sample['wer']:.2f}")
        print(f"SNR: {sample['snr']:.2f} dB")
        print("Original Audio:")
        display(ipd.Audio(sample['original_audio'], rate=16000))
        print("Adversarial Audio:")
        display(ipd.Audio(sample['adversarial_audio'], rate=16000))


=== Epsilon: 0.001 ===
Iteration 0/10, Loss: 15403.3311, SNR: 4.41 dB
Iteration 2/10, Loss: 10776.2949, SNR: 4.41 dB
Iteration 4/10, Loss: 7901.1348, SNR: 4.40 dB
Iteration 6/10, Loss: 6391.1699, SNR: 4.40 dB
Iteration 8/10, Loss: 4918.5762, SNR: 4.40 dB
Processed batch, total samples: 16
Iteration 0/10, Loss: 16429.4453, SNR: 5.28 dB
Iteration 2/10, Loss: 11482.7432, SNR: 5.28 dB
Iteration 4/10, Loss: 8545.3750, SNR: 5.27 dB
Iteration 6/10, Loss: 6495.8047, SNR: 5.27 dB
Iteration 8/10, Loss: 5283.2368, SNR: 5.27 dB
Processed batch, total samples: 32
Iteration 0/10, Loss: 16396.1875, SNR: 5.26 dB
Iteration 2/10, Loss: 12211.1289, SNR: 5.25 dB
Iteration 4/10, Loss: 9692.1719, SNR: 5.25 dB
Iteration 6/10, Loss: 7767.5518, SNR: 5.25 dB
Iteration 8/10, Loss: 6571.8115, SNR: 5.24 dB
Processed batch, total samples: 48
Iteration 0/10, Loss: 12261.5820, SNR: 3.94 dB
Iteration 2/10, Loss: 10247.5723, SNR: 3.94 dB
Iteration 4/10, Loss: 8545.6582, SNR: 3.93 dB
Iteration 6/10, Loss: 7108.7744, SN

Adversarial Audio:



Sample 1:
Ground Truth: REMEMBERING HAS TO BE A PRESENT OCCURRENCE IN SOME WAY RESEMBLING OR RELATED TO WHAT IS REMEMBERED
Original Transcription: REMEMBERING HAS TO BE A PRESENT OCCURRENCE IN SOME WAY RESEMBLING OR RELATED TO WHAT IS REMEMBERED
Adversarial Transcription: ONBIN HOS TO BE POSCURRENTS IN SOME WAY RESEMBLIN ORELATED TO WHAT IS REMEMBERED
CER: 0.24
WER: 0.47
SNR: 4.42 dB
Original Audio:


Adversarial Audio:



Sample 2:
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Original Transcription: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: ALL NOT THINK SUCH AN EFFRENCE IS WARNTED
CER: 0.27
WER: 0.50
SNR: 3.26 dB
Original Audio:


...

Adversarial Audio:



Sample 1:
Ground Truth: REMEMBERING HAS TO BE A PRESENT OCCURRENCE IN SOME WAY RESEMBLING OR RELATED TO WHAT IS REMEMBERED
Original Transcription: REMEMBERING HAS TO BE A PRESENT OCCURRENCE IN SOME WAY RESEMBLING OR RELATED TO WHAT IS REMEMBERED
Adversarial Transcription: OR
CER: 0.98
WER: 0.94
SNR: 2.87 dB
Original Audio:


Adversarial Audio:



Sample 2:
Ground Truth: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Original Transcription: BUT I DO NOT THINK SUCH AN INFERENCE IS WARRANTED
Adversarial Transcription: HEI
CER: 0.94
WER: 1.00
SNR: 1.51 dB
Original Audio:


Adversarial Audio:
